# 9. Gene-Query JEPA IL1B knockout (notebooks 06 + 07)

Same analysis as **06 §6.2–6.3 and 07**, on the JEPA count decoder.

§9.2 pads **IL1B** out of the **90m** source (`lps_90min_perturb`), keeps the same gene-query quiz, and decodes counts twice (control vs KO). That is the JEPA analogue of MaskGIT `val.py`.

| Layer | Meaning |
|---|---|
| `X` / `pert_counts` | KO prediction |
| `pred_counts` | matched control prediction |
| `true_counts` | measured 6h / 10h target |

The DEG is **KO vs control**, not decoder vs measured.


## 9.1. Config (06 §6.1 analogue)

Frozen JEPA winner + CountHead `epoch=39` from the 26 Aug full count run that actually saved a checkpoint. KO dump goes to sod2 `.../count_decoder_runs/il1b_src_90m/` (not `toy_runs/`).


In [33]:
from pathlib import Path
import os

WORKSPACE = Path("/home/stuke1/perturbgen")
REPO = WORKSPACE / "Perturbgen"
SOD2 = Path("/mnt/sod2-project/csb4/stuke1/perturbgen")
TOKENIZED_90M = WORKSPACE / "T_perturb" / "tokenized_data" / "lps_90min_perturb"

GPU = 3  # physical device id passed as --gpu (same default as notebook 06)
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ["WANDB_MODE"] = "disabled"
os.environ.pop("MPLBACKEND", None)

JEPA_CKPT = Path(
    "/mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/full_runs/"
    "encL3_fz5thenU_max256_predL1-6_splitT_ep10/"
    "fzT_thenU5_vic1_contr0.3_q128_predL1/checkpoints/epoch=09.ckpt"
)
COUNT_CKPT = Path(
    "/mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/count_decoder_runs/"
    "countdec_fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs64_ep40_lr0.0001_seed0_splitT_20260826_150037/"
    "checkpoints/epoch=39.ckpt"
)
KO_DIR = SOD2 / "gene_query_jepa" / "count_decoder_runs" / "il1b_src_90m"
KO_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = KO_DIR / "post_analyses"
FIG_DIR.mkdir(parents=True, exist_ok=True)
GENE_NAME_ID_DICT = Path(
    "/mnt/sod2-project/csb4/stuke1/Geneformer/geneformer/gene_name_id_dict.pkl"
)

hits = sorted(KO_DIR.glob("*_minference_adata_gENSG00000125538_ssrc_tmask.h5ad"))
FORCE_KO = True  # re-run dump (06 aggregate); set False to reuse an existing h5ad

print("JEPA_CKPT:", JEPA_CKPT)
print("COUNT_CKPT:", COUNT_CKPT)
print("KO_DIR:", KO_DIR)
print("GPU:", GPU)
print("existing dumps:", [p.name for p in hits])
print("FORCE_KO:", FORCE_KO)
assert JEPA_CKPT.is_file(), JEPA_CKPT
assert COUNT_CKPT.is_file(), COUNT_CKPT
assert TOKENIZED_90M.is_dir(), TOKENIZED_90M


JEPA_CKPT: /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/full_runs/encL3_fz5thenU_max256_predL1-6_splitT_ep10/fzT_thenU5_vic1_contr0.3_q128_predL1/checkpoints/epoch=09.ckpt
COUNT_CKPT: /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/count_decoder_runs/countdec_fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs64_ep40_lr0.0001_seed0_splitT_20260826_150037/checkpoints/epoch=39.ckpt
KO_DIR: /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/count_decoder_runs/il1b_src_90m
GPU: 3
existing dumps: ['20260827-10:04_minference_adata_gENSG00000125538_ssrc_tmask.h5ad', '20260827-10:42_minference_adata_gENSG00000125538_ssrc_tmask.h5ad']
FORCE_KO: True


## 9.2. In-silico IL1B KO (06 §6.2 analogue)

GPU. Pads IL1B (`ENSG00000125538`, global src token via `tokenid_to_rowid`, not the local row id in `token_id_to_genename`) in the 90m source, decodes 6h and 10h with the frozen CountHead. Cells without IL1B in the 90m token list are dropped, same as 06.

Skip if `FORCE_KO` is False (dump already on disk).


In [34]:
import os
import subprocess

if FORCE_KO:
    cmd = [
        "python", "docs/examples/train_gene_query_jepa.py",
        "--eval-count-ko", "true",
        "--jepa-ckpt", str(JEPA_CKPT),
        "--count-ckpt", str(COUNT_CKPT),
        "--tokenized", str(TOKENIZED_90M),
        "--output-dir", str(KO_DIR),
        "--ko-gene", "ENSG00000125538",
        "--ko-pred-tps", "2", "3",
        "--data", "full",
        "--split", "false",
        "--batch-size", "64",
        "--max-len", "256",
        "--gpu", str(GPU),
        "--num-workers", "2",
    ]
    print(" ".join(cmd))
    env = os.environ.copy()
    env.pop("CUDA_VISIBLE_DEVICES", None)
    subprocess.run(cmd, cwd=str(REPO), check=True, env=env)
else:
    print("Skip KO dump; using", hits[-1] if hits else KO_DIR)


python docs/examples/train_gene_query_jepa.py --eval-count-ko true --jepa-ckpt /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/full_runs/encL3_fz5thenU_max256_predL1-6_splitT_ep10/fzT_thenU5_vic1_contr0.3_q128_predL1/checkpoints/epoch=09.ckpt --count-ckpt /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/count_decoder_runs/countdec_fzF_encL3_predL3_q128_mixed_lg1_lc0.1_contr0_vic1_0.04_bs64_ep40_lr0.0001_seed0_splitT_20260826_150037/checkpoints/epoch=39.ckpt --tokenized /home/stuke1/perturbgen/T_perturb/tokenized_data/lps_90min_perturb --output-dir /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/count_decoder_runs/il1b_src_90m --ko-gene ENSG00000125538 --ko-pred-tps 2 3 --data full --split false --batch-size 64 --max-len 256 --gpu 3 --num-workers 2


Seed set to 0


JEPA IL1B KO on cuda:3; src token 1977 (local 284, ENSG00000125538)
Loading 3_10h_LPS.dataset...
Loading 1_normal.dataset...
Loading 2_6h_LPS.dataset...
Loading 3_10h_LPS.h5ad...


/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 2_6h_LPS.h5ad...


/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 1_normal.h5ad...
Start datamodule
Loading frozen JEPA /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/full_runs/encL3_fz5thenU_max256_predL1-6_splitT_ep10/fzT_thenU5_vic1_contr0.3_q128_predL1/checkpoints/epoch=09.ckpt
GeneQueryJEPA: encoder=scmaskgit, d_model=768, n_queries=128 (shared 50% / tgt-only 30% / absent rest), predictor_layers=1, unfreeze_encoder_epochs=0, freeze_encoder_epochs=5, lambda_gene=1.0, lambda_cell=0.1, lambda_contr=0.3, tau=0.1, vicreg_var=1.0, vicreg_cov=0.04
count ckpt head_keys=8 missing_jepa=348 unexpected=0
  batch 0 kept_rows=4
  batch 40 kept_rows=38
  batch 60 kept_rows=66
  batch 160 kept_rows=134
  batch 420 kept_rows=238
  batch 540 kept_rows=296
  batch 620 kept_rows=346
  batch 840 kept_rows=440
  batch 860 kept_rows=454
  batch 960 kept_rows=526
  batch 980 kept_rows=552
  batch 1000 kept_rows=570
  batch 1080 kept_rows=624
  batch 1100 kept_rows=650
  batch 1160 kept_rows=702
  batch 1180 kept_rows=718
  batch 1200 kept_rows=740
  b

/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Wrote /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/count_decoder_runs/il1b_src_90m/20260827-11:03_minference_adata_gENSG00000125538_ssrc_tmask.h5ad  n_obs=20706 x 2000
time_after_LPS
6h_LPS     11751
10h_LPS     8955


## 9.3. Analyse IL1B KO effect (06 §6.3 analogue)

- `adata.X` = KO prediction
- `layers['pred_counts']` = control prediction
- `layers['true_counts']` = measured target (sanity check only)

`delta = mean(KO) − mean(control)` on **CD14 monocytes**, 6h and 10h. Same tables/bars/scatter as 06.3.


In [35]:
import pickle

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display
from scipy import stats

sc.settings.figdir = str(FIG_DIR)

hits = sorted(KO_DIR.glob("*_minference_adata_gENSG00000125538_ssrc_tmask.h5ad"))
assert hits, f"no KO dump under {KO_DIR} — set FORCE_KO=True and run §9.2"
H5AD_PATH = hits[-1]
print("Loading:", H5AD_PATH)
adata = ad.read_h5ad(H5AD_PATH)
adata.layers["pert_counts"] = np.asarray(adata.X).copy()
print(adata)
print(adata.obs["time_after_LPS"].value_counts())

with open(GENE_NAME_ID_DICT, "rb") as f:
    symbol_to_ensembl = pickle.load(f)
ens2sym = {v: k for k, v in symbol_to_ensembl.items()}
adata.var["gene_symbol"] = [ens2sym.get(g, g) for g in adata.var_names.astype(str)]
if "ENSG00000125538" in set(adata.var_names.astype(str)):
    print("IL1B symbol:", adata.var.loc["ENSG00000125538", "gene_symbol"])


Loading: /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/count_decoder_runs/il1b_src_90m/20260827-11:03_minference_adata_gENSG00000125538_ssrc_tmask.h5ad
AnnData object with n_obs × n_vars = 20706 × 2000
    obs: 'cell_type_harmonized', 'cell_pairing_index', 'time_after_LPS', 'cell_idx'
    layers: 'pert_counts', 'pred_counts', 'true_counts'
time_after_LPS
6h_LPS     11751
10h_LPS     8955
Name: count, dtype: int64
IL1B symbol: IL1B


Per-gene KO table, same layout as 06.3. Default subset: **CD14 monocytes**.


In [36]:
def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=np.float64)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = ranked * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n, dtype=np.float64)
    out[order] = np.clip(q, 0, 1)
    return out


def effect_table(adata, time_label, cell_type="CD14 monocytes", eps=1e-3):
    mask = adata.obs["time_after_LPS"].astype(str) == time_label
    if cell_type is not None:
        mask &= adata.obs["cell_type_harmonized"].astype(str) == cell_type
    sub = adata[mask]
    if sub.n_obs == 0:
        raise ValueError(f"No cells for time={time_label}, cell_type={cell_type}")
    pert = np.asarray(sub.layers["pert_counts"], dtype=np.float64)
    ctrl = np.asarray(sub.layers["pred_counts"], dtype=np.float64)
    true = np.asarray(sub.layers["true_counts"], dtype=np.float64)
    mean_pert = pert.mean(axis=0)
    mean_ctrl = ctrl.mean(axis=0)
    delta = mean_pert - mean_ctrl
    log2fc = np.log2(mean_pert + eps) - np.log2(mean_ctrl + eps)
    _, pvals = stats.ttest_rel(pert, ctrl, axis=0, nan_policy="omit")
    pvals = np.nan_to_num(pvals, nan=1.0)
    fdr = bh_fdr(pvals)
    n_sample = min(sub.n_obs, 500)
    rng = np.random.default_rng(0)
    idx = rng.choice(sub.n_obs, size=n_sample, replace=False)
    rs = []
    for i in idx:
        if true[i].std() > 0 and ctrl[i].std() > 0:
            rs.append(stats.pearsonr(ctrl[i], true[i])[0])
    df = pd.DataFrame(
        {
            "ensembl_id": sub.var_names.astype(str),
            "gene_symbol": sub.var["gene_symbol"].astype(str).values,
            "mean_pert": mean_pert,
            "mean_ctrl": mean_ctrl,
            "mean_true": true.mean(axis=0),
            "delta": delta,
            "log2fc": log2fc,
            "pvalue": pvals,
            "fdr": fdr,
        }
    ).sort_values("delta")
    df.attrs["n_cells"] = int(sub.n_obs)
    df.attrs["mean_ctrl_vs_true_pearson"] = float(np.nanmean(rs))
    return df


FOCUS_CELL_TYPE = "CD14 monocytes"
tables = {}
times = [t for t in ["6h_LPS", "10h_LPS"] if t in set(adata.obs["time_after_LPS"].astype(str))]
for tp in times:
    df = effect_table(adata, tp, cell_type=FOCUS_CELL_TYPE)
    tables[tp] = df
    tag = FOCUS_CELL_TYPE.replace(" ", "_")
    out_csv = KO_DIR / f"il1b_ko_effect_{tp}_{tag}.csv"
    df.to_csv(out_csv, index=False)
    print(f"{tp}: n_cells={df.attrs['n_cells']}, ctrl vs true Pearson~{df.attrs['mean_ctrl_vs_true_pearson']:.3f}")
    print("Most down after KO:")
    display(df.head(8)[["gene_symbol", "delta", "log2fc", "fdr"]])
    print("Most up after KO:")
    display(df.tail(8)[["gene_symbol", "delta", "log2fc", "fdr"]].iloc[::-1])


6h_LPS: n_cells=10402, ctrl vs true Pearson~0.871
Most down after KO:


,gene_symbol,delta,log2fc,fdr
122,S100A8,-0.180519,-0.001480,6.868832e-39
120,S100A9,-0.098528,-0.001132,8.567490e-20
1300,LYZ,-0.049074,-0.001787,1.342104e-15
1168,MALAT1,-0.016053,-0.000387,4.278490e-05
651,VCAN,-0.015480,-0.002402,9.312696e-78
1277,TUBA1A,-0.010765,-0.002359,6.761307e-80
117,S100A10,-0.009162,-0.000769,2.518389e-19
975,ANXA1,-0.008374,-0.001477,4.421196e-27


Most up after KO:


,gene_symbol,delta,log2fc,fdr
801,ACTB,0.120773,0.001264,1.672794e-85
1772,FTL,0.100281,0.001432,1.398721e-48
1153,FTH1,0.085067,0.001213,7.606662e-56
150,FCER1G,0.032545,0.002512,2.975194e-73
124,S100A4,0.030080,0.001019,2.298170e-40
1741,TYROBP,0.028680,0.002078,1.843947e-103
754,EEF1A1,0.027882,0.001060,1.009198e-28
1550,COTL1,0.027432,0.003137,3.955891e-61


10h_LPS: n_cells=7822, ctrl vs true Pearson~0.837
Most down after KO:


,gene_symbol,delta,log2fc,fdr
122,S100A8,-0.036066,-0.001136,6.203800e-38
120,S100A9,-0.032602,-0.001104,1.768053e-22
1300,LYZ,-0.017382,-0.001165,8.818413e-12
1277,TUBA1A,-0.007385,-0.001775,3.156065e-68
651,VCAN,-0.004123,-0.001915,2.665843e-58
1167,NEAT1,-0.002937,-0.000645,1.843154e-16
117,S100A10,-0.002867,-0.000310,7.313814e-09
121,S100A12,-0.002602,-0.000574,1.181032e-11


Most up after KO:


,gene_symbol,delta,log2fc,fdr
801,ACTB,0.047866,0.000777,4.493472e-33
1772,FTL,0.037339,0.000817,1.396829e-33
1153,FTH1,0.031990,0.000931,6.997262e-46
754,EEF1A1,0.017652,0.000969,3.164147e-46
124,S100A4,0.013476,0.000620,1.699050e-30
150,FCER1G,0.010986,0.001467,1.535118e-53
1741,TYROBP,0.009444,0.001162,2.245442e-76
688,CD74,0.006881,0.000425,4.751613e-02


### Top predicted KO shifts

06 labels: Down / Up after KO. Same here.


In [20]:
def plot_top_effects(df, title, n=15, path=None):
    down = df.nsmallest(n, "delta")
    up = df.nlargest(n, "delta")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=False)
    for ax, part, color, lab in [
        (axes[0], down.iloc[::-1], "#b2182b", "Down after KO"),
        (axes[1], up.iloc[::-1], "#2166ac", "Up after KO"),
    ]:
        ax.barh(part["gene_symbol"], part["delta"], color=color)
        ax.axvline(0, color="k", lw=0.8)
        ax.set_xlabel("mean(pert) - mean(control)")
        ax.set_title(lab)
    fig.suptitle(title)
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, dpi=150, bbox_inches="tight")
        print("saved", path)
    plt.show()

tag = FOCUS_CELL_TYPE.replace(" ", "_")
for tp, df in tables.items():
    plot_top_effects(
        df,
        title=f"JEPA IL1B KO effect @ {tp} ({FOCUS_CELL_TYPE})",
        path=KO_DIR / f"il1b_ko_top_genes_{tp}_{tag}.png",
    )


saved /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/count_decoder_runs/il1b_src_90m/il1b_ko_top_genes_6h_LPS_CD14_monocytes.png
saved /mnt/sod2-project/csb4/stuke1/perturbgen/gene_query_jepa/count_decoder_runs/il1b_src_90m/il1b_ko_top_genes_10h_LPS_CD14_monocytes.png


### Control prediction sanity check

How close is the **unperturbed** prediction to measured counts? Not the KO.


In [21]:
n_ax = max(len(times), 1)
fig, axes = plt.subplots(1, n_ax, figsize=(4.2 * n_ax, 4))
if n_ax == 1:
    axes = [axes]
for ax, tp in zip(axes, times):
    mask = adata.obs["time_after_LPS"].astype(str) == tp
    if FOCUS_CELL_TYPE is not None:
        mask &= adata.obs["cell_type_harmonized"].astype(str) == FOCUS_CELL_TYPE
    sub = adata[mask]
    ctrl = np.asarray(sub.layers["pred_counts"]).mean(axis=0)
    true = np.asarray(sub.layers["true_counts"]).mean(axis=0)
    r = stats.pearsonr(ctrl, true)[0]
    ax.scatter(true, ctrl, s=8, alpha=0.35, c="#444444")
    lims = [0, max(float(true.max()), float(ctrl.max()))]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set_xlabel("mean true counts")
    ax.set_ylabel("mean control pred")
    ax.set_title(f"{tp}\nPearson r={r:.3f} (n={sub.n_obs})")
fig.suptitle("JEPA control prediction vs measured target (not KO)")
fig.tight_layout()
fig.savefig(KO_DIR / f"il1b_control_vs_true_{tag}.png", dpi=150, bbox_inches="tight")
plt.show()
il1b = "ENSG00000125538"
for tp, df in tables.items():
    hit = df.loc[df["ensembl_id"] == il1b]
    if len(hit):
        row = hit.iloc[0]
        print(f"IL1B KO @ {tp}: delta={row['delta']:.4f}, log2fc={row['log2fc']:.4f}, fdr={row['fdr']:.2e}")


IL1B KO @ 6h_LPS: delta=-0.0009, log2fc=-0.0027, fdr=2.98e-46
IL1B KO @ 10h_LPS: delta=-0.0002, log2fc=-0.0010, fdr=2.79e-13


## 9.4. Post analysis (notebook 07 analogue)

Concat `status=perturbed` from `X` and `status=unperturbed` from `pred_counts`. Then the same `normalize_total` + `log1p`, Wilcoxon, myeloid, Enrichr as 07.


In [24]:
pred = adata.copy()
pred.X = pred.layers["pert_counts"].copy()
ctrl = adata.copy()
ctrl.X = ctrl.layers["pred_counts"].copy()
pred.obs["status"] = "perturbed"
ctrl.obs["status"] = "unperturbed"
pred.obs["status_time"] = "perturbed_" + pred.obs["time_after_LPS"].astype(str)
ctrl.obs["status_time"] = "unperturbed_" + ctrl.obs["time_after_LPS"].astype(str)
adata_both = ad.concat({"perturbed": pred, "unperturbed": ctrl}, label="batch_status", index_unique="-")
print(adata_both)
print(adata_both.obs["status"].value_counts())


AnnData object with n_obs × n_vars = 41412 × 2000
    obs: 'cell_type_harmonized', 'cell_pairing_index', 'time_after_LPS', 'cell_idx', 'status', 'status_time', 'batch_status'
    layers: 'pert_counts', 'pred_counts', 'true_counts'
status
perturbed      20706
unperturbed    20706
Name: count, dtype: int64


Raw-scale counts. Same two lines as notebook 07 before DEG.


In [25]:
sc.pp.normalize_total(adata_both)
sc.pp.log1p(adata_both)


Cell-type names, same map as 05 / 07 / 08.


In [26]:
ct_map = {
    "B cell": "B cells",
    "CD14 monocytes": "CD14+ monocytes",
    "CD16 monocytes": "CD16+ monocytes",
    "CD4+ T cells": "CD4+ T cells",
    "CD8+ T cells": "CD8+ T cells",
    "Dendritic cells": "Dendritic cells",
    "NK": "NK cells",
    "NKT": "NKT cells",
    "Plasmocytoid dendritic cell": "Plasmacytoid dendritic cells",
    "hematopoietic stem cell": "Hematopoietic stem cells",
    "platelet": "Platelets",
}
col = adata_both.obs["cell_type_harmonized"]
if str(col.dtype) == "category":
    adata_both.obs["cell_type_harmonized"] = col.cat.rename_categories(
        {c: ct_map.get(c, c) for c in col.cat.categories}
    )
else:
    adata_both.obs["cell_type_harmonized"] = col.astype(str).map(lambda x: ct_map.get(x, x))


### DEG counts by cell type (07 bar plot)

Wilcoxon FDR < 0.05 for **perturbed vs unperturbed** (IL1B KO).


In [27]:
import seaborn as sns

sns.set_context("talk", font_scale=1.2)
cell_types = adata_both.obs["cell_type_harmonized"].unique()
results = []
for tp in times:
    pert = f"perturbed_{tp}"
    unpert = f"unperturbed_{tp}"
    for ct in cell_types:
        mask_p = (adata_both.obs["status_time"] == pert) & (adata_both.obs["cell_type_harmonized"] == ct)
        mask_m = (adata_both.obs["status_time"] == unpert) & (adata_both.obs["cell_type_harmonized"] == ct)
        if mask_p.sum() == 0 or mask_m.sum() == 0:
            continue
        sub = adata_both[mask_p | mask_m].copy()
        sub.obs["group"] = "unperturbed"
        sub.obs.loc[sub.obs["status_time"] == pert, "group"] = "perturbed"
        sub.obs["group"] = sub.obs["group"].astype("category")
        sc.tl.rank_genes_groups(sub, groupby="group", reference="unperturbed", method="wilcoxon")
        degs = sc.get.rank_genes_groups_df(sub, group="perturbed")
        results.append({"cell_type": ct, "timepoint": tp, "deg_count": int((degs["pvals_adj"] < 0.05).sum())})
df = pd.DataFrame(results).sort_values("deg_count", ascending=False)
df["timepoint"] = df["timepoint"].replace({"6h_LPS": "6h", "10h_LPS": "10h"})
df.to_csv(FIG_DIR / "deg_counts_by_celltype.csv", index=False)
plt.figure(figsize=(8, 6))
sns.barplot(data=df, y="cell_type", x="deg_count", hue="timepoint")
plt.ylabel("Cell type")
plt.xlabel("DEGs count")
plt.title("Perturbation effect (IL1B KO)")
plt.tight_layout()
plt.savefig(FIG_DIR / "deg_counts_by_celltype.png", dpi=150, bbox_inches="tight")
plt.show()
display(df)


,cell_type,timepoint,deg_count
4,CD16+ monocytes,6h,765
0,B cells,6h,0
1,CD4+ T cells,6h,0
2,CD8+ T cells,6h,0
3,CD14+ monocytes,6h,0
5,Dendritic cells,6h,0
6,NK cells,6h,0
7,NKT cells,6h,0
8,Plasmacytoid dendritic cells,6h,0
9,Hematopoietic stem cells,6h,0


### Myeloid subset (07)

CD14+ monocytes, CD16+ monocytes, dendritic cells.


In [30]:
MYELOID = ["CD14+ monocytes", "Dendritic cells", "CD16+ monocytes"]
adata_mye = adata_both[adata_both.obs["cell_type_harmonized"].isin(MYELOID)].copy()
print(adata_mye)
print(adata_mye.obs["status_time"].value_counts())


AnnData object with n_obs × n_vars = 38790 × 2000
    obs: 'cell_type_harmonized', 'cell_pairing_index', 'time_after_LPS', 'cell_idx', 'status', 'status_time', 'batch_status'
    uns: 'log1p'
    layers: 'pert_counts', 'pred_counts', 'true_counts'
status_time
perturbed_6h_LPS       11072
unperturbed_6h_LPS     11072
perturbed_10h_LPS       8323
unperturbed_10h_LPS     8323
Name: count, dtype: int64


Ensembl → symbol for Enrichr. Rebuild from `ens2sym` (`ad.concat` drops `.var`).


In [29]:
adata_mye.var["ensembl_id"] = adata_mye.var_names.astype(str)
adata_mye.var["gene_symbol"] = adata_mye.var["ensembl_id"].map(ens2sym)
n_missing = int(adata_mye.var["gene_symbol"].isna().sum())
print(f"Unmapped genes: {n_missing}/{adata_mye.n_vars}")
adata_mye = adata_mye[:, adata_mye.var["gene_symbol"].notna()].copy()
adata_mye.var_names = adata_mye.var["gene_symbol"].astype(str)
adata_mye.var_names_make_unique()
print("IL1B in var_names:", "IL1B" in set(adata_mye.var_names))


Unmapped genes: 1/2000
IL1B in var_names: True


Wilcoxon inside myeloid, pert vs unpert, per time.


In [31]:
deg_results = {}
for tp, tps in [("6h", "6h_LPS"), ("10h", "10h_LPS")]:
    pert = f"perturbed_{tps}"
    unpert = f"unperturbed_{tps}"
    if pert not in set(adata_mye.obs["status_time"].astype(str)):
        continue
    ad_sub = adata_mye[adata_mye.obs["status_time"].isin([pert, unpert])].copy()
    ad_sub.obs["cond"] = ad_sub.obs["status_time"].map({pert: "pert", unpert: "ctrl"}).astype("category")
    sc.tl.rank_genes_groups(
        ad_sub, groupby="cond", groups=["pert"], reference="ctrl",
        method="wilcoxon", corr_method="benjamini-hochberg",
    )
    de = sc.get.rank_genes_groups_df(ad_sub, group="pert")
    de.to_csv(FIG_DIR / f"myeloid_degs_{tp}.csv", index=False)
    sig = de[de["pvals_adj"] < 0.05]
    up = sig[sig["logfoldchanges"] > 0]
    down = sig[sig["logfoldchanges"] < 0]
    deg_results[tp] = {"up": up["names"].tolist(), "down": down["names"].tolist()}
    print(f"{tp}: n={ad_sub.n_obs}, sig={len(sig)}, up={len(up)}, down={len(down)}")


6h: n=22144, sig=0, up=0, down=0
10h: n=16646, sig=0, up=0, down=0


### Enrichr (07)

Same libraries. Gene lists are myeloid **KO DEGs**. Needs network.


In [32]:
import gseapy as gp

PREFERRED_LIBS = {
    "reactome": ["Reactome_Pathways_2024", "Reactome_2022"],
    "go_bp": ["GO_Biological_Process_2025", "GO_Biological_Process_2023"],
}
try:
    available = set(gp.get_library_name(organism="Human"))
    gene_sets = [next(lib for lib in libs if lib in available) for libs in PREFERRED_LIBS.values()]
except Exception as e:
    print(f"Could not list Enrichr libraries ({e}); using defaults")
    gene_sets = ["Reactome_2022", "GO_Biological_Process_2023"]
print("Using gene sets:", gene_sets)

for tp, res in deg_results.items():
    for label, genes in res.items():
        if len(genes) == 0:
            print(f"Skip {tp} {label}: empty gene list")
            continue
        try:
            enr = gp.enrichr(gene_list=genes, gene_sets=gene_sets, organism="Human", outdir=None, background=list(adata_mye.var_names))
        except Exception as e:
            print(f"{tp} {label}: background mode failed ({e}); retrying without")
            try:
                enr = gp.enrichr(gene_list=genes, gene_sets=gene_sets, organism="Human", outdir=None)
            except Exception as e2:
                print(f"{tp} {label}: enrichr failed ({e2}); skip")
                continue
        if enr is None or enr.results is None or len(enr.results) == 0:
            print(f"{tp} {label}: no enrichment results")
            continue
        df_enr = enr.results.sort_values("Adjusted P-value").head(15).copy()
        df_enr.to_csv(FIG_DIR / f"enrichr_{tp}_{label}.csv", index=False)
        df_plot = df_enr.iloc[::-1]
        neglogp = -np.log10(df_plot["Adjusted P-value"].clip(lower=1e-300))
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.barh(df_plot["Term"].str.slice(0, 60), neglogp, color="#2166ac" if label == "up" else "#b2182b")
        ax.axvline(-np.log10(0.05), color="k", ls="--", lw=0.8)
        ax.set_xlabel(r"$-\log_{10}$ adjusted P-value")
        ax.set_title(f"IL1B KO | {tp} {label.upper()}")
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"enrichr_{tp}_{label}.png", dpi=150, bbox_inches="tight")
        plt.show()


Using gene sets: ['Reactome_Pathways_2024', 'GO_Biological_Process_2025']
Skip 6h up: empty gene list
Skip 6h down: empty gene list
Skip 10h up: empty gene list
Skip 10h down: empty gene list
